## *For up-to-date plotting script use plots/combineFiles.ipynb*

In [ ]:
import pandas as pd
import numpy as np
from coffea import util
import itertools
import os, sys
import matplotlib.pyplot as plt
import mplhep as hep
import uproot
hep.style.use("CMS")

sys.path.append('../python/')
from functions import loadCoffeaFile, getLabelMap, getCoffeaFilenames, plotBackgroundEstimate, getHist


## Scale factors and IOV

In [ ]:
# IOVs = ['2016all', '2016APV', '2016']#, '2017']
IOVs = ['2016all']

variable = 'ttbarmass'  # Change this at will
signal = 'ZPrime30'

lumi = {
    "2016APV": 19800.,
    "2016": 16120., #35920 - 19800
    "2016all": 35920,
    "2017": 41530.,
    "2018": 59740./10. #Blinding
}

t_BR = 0.6741
ttbar_BR = 0.4544 #PDG 2019
ttbar_xs1 = 831.76 * (0.09210) #pb For ttbar mass from 700 to 1000
ttbar_xs2 = 831.76 * (0.02474) #pb For ttbar mass from 1000 to Inf
toptag_sf = 0.9
toptag_kf = 1.0 #0.7
qcd_xs = 1370000000.0 #pb From https://cms-gen-dev.cern.ch/xsdb



## analysis categories

In [ ]:
# analysis categories #

label_dict =  getLabelMap()
label_to_int_dict = {label: i for i, label in label_dict.items()}

signal_cats = [ i for i, label in label_dict.items() if '2t' in label]
pretag_cats = [ i for i, label in label_dict.items() if 'pre' in label]


## make plot image filenames

In [ ]:
directories = [
    'images/png/closureTest/2016all',
    'images/png/closureTest/2016APV',
    'images/png/closureTest/2016',
    'images/png/closureTest/2017',
    'images/png/closureTest/2018',
    'images/pdf/closureTest/2016all',
    'images/pdf/closureTest/2016APV',
    'images/pdf/closureTest/2016',
    'images/pdf/closureTest/2017',
    'images/pdf/closureTest/2018',
    'images/png/kinematics/2016all',
    'images/png/kinematics/2016APV',
    'images/png/kinematics/2016',
    'images/png/kinematics/2017',
    'images/png/kinematics/2018',
    'images/pdf/kinematics/2016all',
    'images/pdf/kinematics/2016APV',
    'images/pdf/kinematics/2016',
    'images/pdf/kinematics/2017',
    'images/pdf/kinematics/2018',
]


for path in directories:
    if not os.path.exists(path):
        os.makedirs(path)

In [ ]:
# B = util.load('../outputs/JetHT_2017B_bkgest.coffea')
# C = util.load('../outputs/JetHT_2016APVC.coffea')
# D = util.load('../outputs/JetHT_2016APVD.coffea')
# E = util.load('../outputs/JetHT_2016APVE.coffea')
# Fapv = util.load('../outputs/JetHT_2016APVF.coffea')
# F = util.load('../outputs/JetHT_2016F.coffea')
# G = util.load('../outputs/JetHT_2016G.coffea')
# H = util.load('../outputs/JetHT_2016H.coffea')
# print(B['cutflow']['all events'])
# print(C['cutflow']['all events'])
# print(D['cutflow']['all events'])
# print(E['cutflow']['all events'])
# print(Fapv['cutflow']['all events'])
# print(F['cutflow']['all events'])
# print(G['cutflow']['all events'])
# print(H['cutflow']['all events'])

## functions

In [ ]:
def getHist(hname, ds, bkgest, year, sum_axes=[], integrate_axes={}, masspoint=''):
    
    ######################################################################################
    # hname = histogram name (example: 'ttbarmass')                                      #
    # ds = dataset name (example: 'JetHT')                                               #
    # bkgest = boolean, True if bkg estimate applied                                     #
    # year = '2016APV' or '2016' or '2017' or '2018'                                     #
    # sum_axes = names of axes to sum over for scikit-hep/hist histogram                 #
    # integrate_axes = range to integrate over axis (example: {'anacat': [0,1,2,3,4,5]}) #
    ######################################################################################    

    
    # load histograms and get scale factors
    coffeaFiles = getCoffeaFilenames()
    
    cfiles = []
    sf = []
    bkgest_str = np.where([bkgest], 'weighted', 'unweighted')[0]
    
    for key, file in coffeaFiles[ds][bkgest_str][year].items():
        if masspoint != '':
            if masspoint in key:
                loaded_file = util.load(file)
#                 print(file + ' loaded')
                sum_axes_dict = {ax:sum for ax in sum_axes}
                histo = loaded_file[hname][integrate_axes][sum_axes_dict]
                histo = histo * (lumi[IOV] * 1.0 / loaded_file['cutflow']['sumw'])
                return histo
            
        loaded_file = util.load(file)
        cfiles.append(loaded_file)
        
        
        if 'TTbar' in ds and '700to1000' in key:
            sf.append(lumi[year] * ttbar_xs1 * toptag_sf**2 / loaded_file['cutflow']['sumw'])
        elif 'TTbar' in ds and '1000toInf' in key:
            sf.append(lumi[year] * ttbar_xs2 * toptag_sf**2 / loaded_file['cutflow']['sumw'])     
        elif 'QCD' in ds:
            sf.append(lumi[IOV] * qcd_xs / loaded_file['cutflow']['sumw'])  
        else:
            sf.append(1.)
        
#             if year == '2017':

#         for cfile in cfiles:
#             print(cfile[hname][{'anacat':pretag_cats, 'systematic':'nominal'}]['systematic'])
    
    # sum or integrate axes for all hists from dataset eras or pt bins
    sum_axes_dict = {ax:sum for ax in sum_axes}
#     print(sum_axes_dict)
    
    hists = []
    for cfile in cfiles:
        hists.append(cfile[hname][integrate_axes][sum_axes_dict])
#         print(cfile[hname][integrate_axes][sum_axes_dict])
    
    
    # sum all hists from dataset eras or pt bins
    histo = hists[0]*sf[0]
    if len(hists) > 1:
        for i in range(len(hists) - 1): 
            histo = histo + hists[i+1]*sf[i+1]
            
            
    
    return histo
    
            
            
def plotBackgroundEstimate(histname, Hdata, Hntmj, Httbar, Year, Text='', hsig1=None, hsig2=None, hsig3=None, hsig4=None, hsig5=None):
    
#     print(Hntmj)
#     print(Httbar)
    Hbkg = Hntmj + Httbar
    
    fig, (ax1, ax2) = plt.subplots(nrows=2, height_ratios=[3, 1])

    
    hep.cms.label('', data=True, lumi='{0:0.1f}'.format(lumi[Year]/1000.), year=Year, loc=2, fontsize=20, ax=ax1)
    hep.cms.text(Text, loc=2, fontsize=20, ax=ax1)

    hep.histplot(Hdata,  ax=ax1, histtype='errorbar', color='black', label='Data')
    hep.histplot(Hbkg,   ax=ax1, histtype='fill', color='xkcd:pale gold', label='NTMJ')
    hep.histplot(Httbar, ax=ax1, histtype='fill', color='xkcd:deep red', label='TTbar')
    
    Signal = {
        'RSGluon': r'RS$_{KK}$ Gluon',
        'ZPrime10': r'Z ` $10\%$',
        'ZPrime30': r'Z ` $30\%$',
        'ZPrimeDM': r'Z ` DM'
    }
    
#     if hsig1 != None:
#         hep.histplot(hsig1, ax=ax1, histtype='step', label=Signal[signal]+' 1 TeV')
#         hep.histplot(hsig2, ax=ax1, histtype='step', label=Signal[signal]+' 2 TeV')
#         hep.histplot(hsig3, ax=ax1, histtype='step', label=Signal[signal]+' 3 TeV')
#         hep.histplot(hsig4, ax=ax1, histtype='step', label=Signal[signal]+' 4 TeV')
        
    if hsig1 != None:
        hep.histplot(hsig1, ax=ax1, histtype='step', label=Signal['ZPrime10']+' 1 TeV')
        hep.histplot(hsig2, ax=ax1, histtype='step', label=Signal['ZPrime10']+' 2 TeV')
        hep.histplot(hsig3, ax=ax1, histtype='step', label=Signal['ZPrime10']+' 3 TeV')
        hep.histplot(hsig4, ax=ax1, histtype='step', label=Signal['ZPrime10']+' 4 TeV')
        hep.histplot(hsig5, ax=ax1, histtype='step', label=Signal['ZPrime10']+' 5 TeV')

    ratio_plot =  Hdata / Hbkg.values()
    hep.histplot(ratio_plot, ax=ax2, histtype='errorbar', color='black')
    ax2.set_ylim(0,2)
    ax2.axhline(1, color='black', ls='--')
    ax2.set_ylabel('Data/Bkg')

    ax1.legend(fontsize='xx-small')
    ax1.set_yscale('log')
    ax1.set_ylabel('Events')
    ax1.set_xlabel('')
    ax1.set_ylim(1e-2, 1e7)
    ax1.set_xlim(900, 6000)
    ax2.set_xlim(900, 6000)    
    
    if histname == 'jetpt':
        ax1.set_xlim(400, 1400)#2000)
        ax2.set_xlim(400, 1400)#2000)
    elif histname == 'jeteta':
        ax1.set_xlim(-2.4, 2.4)
        ax2.set_xlim(-2.4, 2.4)
    elif histname == 'jetphi':
        ax1.set_xlim(-np.pi, np.pi)
        ax2.set_xlim(-np.pi, np.pi)
    elif histname == 'jetmass':
        ax1.set_xlim(0, 400)#500)
        ax2.set_xlim(0, 400)#500)   
        
        
        
def plotBackgroundEstimateNoData(histname, Hntmj, Httbar, Year, Text='', hsig1=None, hsig2=None, hsig3=None, hsig4=None, hsig5=None):
    
#     print(Hntmj)
#     print(Httbar)
    Hbkg = Hntmj + Httbar
    
    hep.cms.label('', data=True, lumi='{0:0.1f}'.format(lumi[Year]/1000.), year=Year, loc=2, fontsize=20)
    hep.cms.text(Text, loc=2, fontsize=20)

    hep.histplot(Hbkg, histtype='fill', color='xkcd:pale gold', label='NTMJ')
    hep.histplot(Httbar, histtype='fill', color='xkcd:deep red', label='TTbar')
    
    Signal = {
        'RSGluon': r'RS$_{KK}$ Gluon',
        'ZPrime10': r'Z ` $10\%$',
        'ZPrime30': r'Z ` $30\%$',
        'ZPrimeDM': r'Z ` DM'
    }
    
#     if hsig1 != None:
#         hep.histplot(hsig1, ax=ax1, histtype='step', label=Signal[signal]+' 1 TeV')
#         hep.histplot(hsig2, ax=ax1, histtype='step', label=Signal[signal]+' 2 TeV')
#         hep.histplot(hsig3, ax=ax1, histtype='step', label=Signal[signal]+' 3 TeV')
#         hep.histplot(hsig4, ax=ax1, histtype='step', label=Signal[signal]+' 4 TeV')
        
    if hsig1 != None:
        hep.histplot(hsig1, histtype='step', label=Signal['RSGluon']+' 2 TeV')
        hep.histplot(hsig2, histtype='step', label=Signal['ZPrime10']+' 2 TeV')
        hep.histplot(hsig3, histtype='step', label=Signal['ZPrime30']+' 2 TeV')
        hep.histplot(hsig4, histtype='step', label=Signal['ZPrimeDM']+' 2 TeV')

    plt.legend(fontsize='xx-small')
    plt.yscale('log')
    plt.ylabel('Events')
    plt.xlabel('')
    plt.ylim(1e-2, 1e6)
    plt.xlim(900, 6000)
    plt.xlim(900, 6000)    
    

## plot background estimate (inclusive)

In [ ]:
dirname = 'closureTest'
if variable != 'ttbarmass':
    dirname = 'kinematics'

In [ ]:
# if variable == 'ttbarmass':
    
#     for IOV in IOVs:
    
#         if '2016all' in IOV:
#             httbar_apv  = getHist(variable, 'TTbar', False, '2016APV', sum_axes=['anacat'], integrate_axes={'anacat':signal_cats, 'systematic':'nominal'})
#             hcontam_apv = getHist(variable, 'TTbar', True, '2016APV', sum_axes=['anacat'], integrate_axes={'anacat':pretag_cats, 'systematic':'nominal'})
#             hntmj_apv   = getHist(variable, 'JetHT', True, '2016APV', sum_axes=['anacat'], integrate_axes={'anacat':pretag_cats, 'systematic':'nominal'})
#             hdata_apv   = getHist(variable, 'JetHT', False, '2016APV', sum_axes=['anacat'], integrate_axes={'anacat':signal_cats, 'systematic':'nominal'})
#             hsignal1000_apv = getHist(variable, 'RSGluon', False, '2016APV', sum_axes=['anacat'], integrate_axes={'anacat':signal_cats, 'systematic':'nominal'}, masspoint='2000')
#             hsignal2000_apv = getHist(variable, 'ZPrime10', False, '2016APV', sum_axes=['anacat'], integrate_axes={'anacat':signal_cats, 'systematic':'nominal'}, masspoint='2000')
#             hsignal3000_apv = getHist(variable, 'ZPrime30', False, '2016APV', sum_axes=['anacat'], integrate_axes={'anacat':signal_cats, 'systematic':'nominal'}, masspoint='2000')
#             hsignal4000_apv = getHist(variable, 'ZPrimeDM', False, '2016APV', sum_axes=['anacat'], integrate_axes={'anacat':signal_cats, 'systematic':'nominal'}, masspoint='2000')

#             httbar_noapv  = getHist(variable, 'TTbar', False, '2016', sum_axes=['anacat'], integrate_axes={'anacat':signal_cats, 'systematic':'nominal'})
#             hcontam_noapv = getHist(variable, 'TTbar', True, '2016', sum_axes=['anacat'], integrate_axes={'anacat':pretag_cats, 'systematic':'nominal'})
#             hntmj_noapv   = getHist(variable, 'JetHT', True, '2016', sum_axes=['anacat'], integrate_axes={'anacat':pretag_cats, 'systematic':'nominal'})
#             hdata_noapv   = getHist(variable, 'JetHT', False, '2016', sum_axes=['anacat'], integrate_axes={'anacat':signal_cats, 'systematic':'nominal'})
#             hsignal1000_noapv = getHist(variable, 'RSGluon', False, '2016', sum_axes=['anacat'], integrate_axes={'anacat':signal_cats, 'systematic':'nominal'}, masspoint='2000')
#             hsignal2000_noapv = getHist(variable, 'ZPrime10', False, '2016', sum_axes=['anacat'], integrate_axes={'anacat':signal_cats, 'systematic':'nominal'}, masspoint='2000')
#             hsignal3000_noapv = getHist(variable, 'ZPrime30', False, '2016', sum_axes=['anacat'], integrate_axes={'anacat':signal_cats, 'systematic':'nominal'}, masspoint='2000')
#             hsignal4000_noapv = getHist(variable, 'ZPrimeDM', False, '2016', sum_axes=['anacat'], integrate_axes={'anacat':signal_cats, 'systematic':'nominal'}, masspoint='2000')

#             httbar  = httbar_apv + httbar_noapv
#             hcontam = hcontam_apv + hcontam_noapv
#             hntmj   = hntmj_apv + hntmj_noapv
#             hdata   = hdata_apv + hdata_noapv
#             hsignal1000 = hsignal1000_apv + hsignal1000_noapv
#             hsignal2000 = hsignal2000_apv + hsignal2000_noapv
#             hsignal3000 = hsignal3000_apv + hsignal3000_noapv
#             hsignal4000 = hsignal4000_apv + hsignal4000_noapv

#             hntmj_fixed = hntmj + -1*hcontam
            

#         else:

#             httbar  = getHist(variable, 'TTbar', False, IOV, sum_axes=['anacat'], integrate_axes={'anacat':signal_cats, 'systematic':'nominal'})
#             hcontam = getHist(variable, 'TTbar', True, IOV, sum_axes=['anacat'], integrate_axes={'anacat':pretag_cats, 'systematic':'nominal'})
#             hntmj   = getHist(variable, 'JetHT', True, IOV, sum_axes=['anacat'], integrate_axes={'anacat':pretag_cats, 'systematic':'nominal'})
#             hdata   = getHist(variable, 'JetHT', False, IOV, sum_axes=['anacat'], integrate_axes={'anacat':signal_cats, 'systematic':'nominal'})
            
#             hsignal1000 = getHist(variable, 'RSGluon', False, IOV, sum_axes=['anacat'], integrate_axes={'anacat':signal_cats, 'systematic':'nominal'}, masspoint='2000')
#             hsignal2000 = getHist(variable, 'ZPrime10', False, IOV, sum_axes=['anacat'], integrate_axes={'anacat':signal_cats, 'systematic':'nominal'}, masspoint='2000')
#             hsignal3000 = getHist(variable, 'ZPrime30', False, IOV, sum_axes=['anacat'], integrate_axes={'anacat':signal_cats, 'systematic':'nominal'}, masspoint='2000')
#             hsignal4000 = getHist(variable, 'ZPrimeDM', False, IOV, sum_axes=['anacat'], integrate_axes={'anacat':signal_cats, 'systematic':'nominal'}, masspoint='2000')

#             hntmj_fixed = hntmj + -1*hcontam
            
# #         text = 'data/simulation'+'\n'+r'$\Delta y$ inclusive'+'\n'+r'b tag inclusive'
#         text = 'Preliminary'+'\n'+r'$\Delta y$ inclusive'+'\n'+r'b tag inclusive'
            
#         plotBackgroundEstimate(variable, hdata, hntmj_fixed, httbar, IOV, text, hsignal1000, hsignal2000, hsignal3000, hsignal4000)
# #         if '2016all' in IOV:
# #             plotBackgroundEstimate(variable, hdata, hntmj_fixed, httbar, IOV, text, hsignal1000, hsignal2000, hsignal3000, hsignal4000)
# #         else:
# #             plotBackgroundEstimate(variable, hdata, hntmj_fixed, httbar, IOV, text)
        

#         plt.savefig(f'images/png/{dirname}/{IOV}/closuretest_inclusive.png')
#         plt.savefig(f'images/pdf/{dirname}/{IOV}/closuretest_inclusive.pdf')
#         print(f'images/png/{dirname}/{IOV}/closuretest_inclusive.png saved\n')
# else:
    
#     for IOV in IOVs:
        
#         if '2016all' in IOV:
#             httbar_apv  = getHist(variable, 'TTbar', False, '2016APV', sum_axes=['anacat'], integrate_axes={'anacat':signal_cats})
#             hcontam_apv = getHist(variable, 'TTbar', True, '2016APV', sum_axes=['anacat'], integrate_axes={'anacat':pretag_cats})
#             hntmj_apv   = getHist(variable, 'JetHT', True, '2016APV', sum_axes=['anacat'], integrate_axes={'anacat':pretag_cats})
#             hdata_apv   = getHist(variable, 'JetHT', False, '2016APV', sum_axes=['anacat'], integrate_axes={'anacat':signal_cats})

#             httbar_noapv  = getHist(variable, 'TTbar', False, '2016', sum_axes=['anacat'], integrate_axes={'anacat':signal_cats})
#             hcontam_noapv = getHist(variable, 'TTbar', True, '2016', sum_axes=['anacat'], integrate_axes={'anacat':pretag_cats})
#             hntmj_noapv   = getHist(variable, 'JetHT', True, '2016', sum_axes=['anacat'], integrate_axes={'anacat':pretag_cats})
#             hdata_noapv   = getHist(variable, 'JetHT', False, '2016', sum_axes=['anacat'], integrate_axes={'anacat':signal_cats})

#             httbar  = httbar_apv + httbar_noapv
#             hcontam = hcontam_apv + hcontam_noapv
#             hntmj   = hntmj_apv + hntmj_noapv
#             hdata   = hdata_apv + hdata_noapv

#             hntmj_fixed = hntmj + -1*hcontam

#         else:

#             httbar  = getHist(variable, 'TTbar', False, IOV, sum_axes=['anacat'], integrate_axes={'anacat':signal_cats})
#             hcontam = getHist(variable, 'TTbar', True, IOV, sum_axes=['anacat'], integrate_axes={'anacat':pretag_cats})
#             hntmj   = getHist(variable, 'JetHT', True, IOV, sum_axes=['anacat'], integrate_axes={'anacat':pretag_cats})
#             hdata   = getHist(variable, 'JetHT', False, IOV, sum_axes=['anacat'], integrate_axes={'anacat':signal_cats})

#             hntmj_fixed = hntmj + -1*hcontam
        
#         text = 'Preliminary'+'\n'+r'$\Delta y$ inclusive'+'\n'+r'b tag inclusive'
# #         text = 'data/simulation'+'\n'+r'$\Delta y$ inclusive'+'\n'+r'b tag inclusive'

#         plotBackgroundEstimate(variable, hdata, hntmj_fixed, httbar, IOV, text)

# #         plt.savefig(f'images/png/{dirname}/{IOV}/{variable}_inclusive.png')
# #         plt.savefig(f'images/pdf/{dirname}/{IOV}/{variable}_inclusive.pdf')
        
#         print(f'images/png/{dirname}/{IOV}/{variable}_inclusive.png saved\n')

#     plt.show()


In [ ]:
## plot background estimate (by category)

In [ ]:
cats = ['0bcen', '0bfwd', '1bcen', '1bfwd', '2bcen', '2bfwd']

# cats = ['2bcen', '2bfwd']

if variable == 'ttbarmass':
    
    for IOV in IOVs:
        
        for cat in cats:
            
            signal_cat = label_to_int_dict['2t'+cat]
            pretag_cat = label_to_int_dict['pret'+cat]
            
            if '2016all' in IOV:
                
                httbar_apv = getHist('ttbarmass', 'TTbar', False, '2016APV', sum_axes=[], integrate_axes={'anacat':signal_cat, 'systematic':'nominal'})
                hcontam_apv = getHist('ttbarmass', 'TTbar', True, '2016APV', sum_axes=[], integrate_axes={'anacat':pretag_cat, 'systematic':'nominal'})
                hntmj_apv = getHist('ttbarmass', 'JetHT', True, '2016APV',   sum_axes=[], integrate_axes={'anacat':pretag_cat, 'systematic':'nominal'})
                hdata_apv = getHist('ttbarmass', 'JetHT', False, '2016APV',  sum_axes=[], integrate_axes={'anacat':signal_cat, 'systematic':'nominal'})

                httbar_noapv = getHist('ttbarmass', 'TTbar', False, '2016', sum_axes=[], integrate_axes={'anacat':signal_cat, 'systematic':'nominal'})
                hcontam_noapv = getHist('ttbarmass', 'TTbar', True, '2016', sum_axes=[], integrate_axes={'anacat':pretag_cat, 'systematic':'nominal'})
                hntmj_noapv = getHist('ttbarmass', 'JetHT', True, '2016',   sum_axes=[], integrate_axes={'anacat':pretag_cat, 'systematic':'nominal'})
                hdata_noapv = getHist('ttbarmass', 'JetHT', False, '2016',  sum_axes=[], integrate_axes={'anacat':signal_cat, 'systematic':'nominal'})

                hsignal1000_apv = getHist(variable, 'ZPrime10', False, '2016APV', sum_axes=[], integrate_axes={'anacat':signal_cat, 'systematic':'nominal'}, masspoint='1000')
                hsignal2000_apv = getHist(variable, 'ZPrime10', False, '2016APV', sum_axes=[], integrate_axes={'anacat':signal_cat, 'systematic':'nominal'}, masspoint='2000')
                hsignal3000_apv = getHist(variable, 'ZPrime10', False, '2016APV', sum_axes=[], integrate_axes={'anacat':signal_cat, 'systematic':'nominal'}, masspoint='3000')
                hsignal4000_apv = getHist(variable, 'ZPrime10', False, '2016APV', sum_axes=[], integrate_axes={'anacat':signal_cat, 'systematic':'nominal'}, masspoint='4000')
                hsignal5000_apv = getHist(variable, 'ZPrime10', False, '2016APV', sum_axes=[], integrate_axes={'anacat':signal_cat, 'systematic':'nominal'}, masspoint='5000')

                hsignal1000_noapv = getHist(variable, 'ZPrime10', False, '2016', sum_axes=[], integrate_axes={'anacat':signal_cat, 'systematic':'nominal'}, masspoint='1000')
                hsignal2000_noapv = getHist(variable, 'ZPrime10', False, '2016', sum_axes=[], integrate_axes={'anacat':signal_cat, 'systematic':'nominal'}, masspoint='2000')
                hsignal3000_noapv = getHist(variable, 'ZPrime10', False, '2016', sum_axes=[], integrate_axes={'anacat':signal_cat, 'systematic':'nominal'}, masspoint='3000')
                hsignal4000_noapv = getHist(variable, 'ZPrime10', False, '2016', sum_axes=[], integrate_axes={'anacat':signal_cat, 'systematic':'nominal'}, masspoint='4000')
                hsignal5000_noapv = getHist(variable, 'ZPrime10', False, '2016', sum_axes=[], integrate_axes={'anacat':signal_cat, 'systematic':'nominal'}, masspoint='5000')

                httbar  = httbar_apv + httbar_noapv
                hcontam = hcontam_apv + hcontam_noapv
                hntmj   = hntmj_apv + hntmj_noapv
                hdata   = hdata_apv + hdata_noapv
                hsignal1000 = hsignal1000_apv + hsignal1000_noapv
                hsignal2000 = hsignal2000_apv + hsignal2000_noapv
                hsignal3000 = hsignal3000_apv + hsignal3000_noapv
                hsignal4000 = hsignal4000_apv + hsignal4000_noapv
                hsignal5000 = hsignal5000_apv + hsignal5000_noapv

                hntmj_fixed = hntmj + -1*hcontam
            

            else:

                httbar  = getHist(variable, 'TTbar', False, IOV, sum_axes=['anacat'], integrate_axes={'anacat':signal_cats, 'systematic':'nominal'})
                hcontam = getHist(variable, 'TTbar', True, IOV, sum_axes=['anacat'], integrate_axes={'anacat':pretag_cats, 'systematic':'nominal'})
                hntmj   = getHist(variable, 'JetHT', True, IOV, sum_axes=['anacat'], integrate_axes={'anacat':pretag_cats, 'systematic':'nominal'})
                hdata   = getHist(variable, 'JetHT', False, IOV, sum_axes=['anacat'], integrate_axes={'anacat':signal_cats, 'systematic':'nominal'})
                
                hsignal1000 = getHist(variable, 'RSGluon', False, IOV, sum_axes=[], integrate_axes={'anacat':signal_cat, 'systematic':'nominal'}, masspoint='2000')
                hsignal2000 = getHist(variable, 'ZPrime10', False, IOV, sum_axes=[], integrate_axes={'anacat':signal_cat, 'systematic':'nominal'}, masspoint='2000')
                hsignal3000 = getHist(variable, 'ZPrime30', False, IOV, sum_axes=[], integrate_axes={'anacat':signal_cat, 'systematic':'nominal'}, masspoint='2000')
                hsignal4000 = getHist(variable, 'ZPrimeDM', False, IOV, sum_axes=[], integrate_axes={'anacat':signal_cat, 'systematic':'nominal'}, masspoint='2000')

                hntmj_fixed = hntmj + -1*hcontam
                        
            dytext = ''
            if 'cen' in cat:
                dytext = r'$\Delta y$ < 1.0'
            elif 'fwd' in cat:
                dytext = r'$\Delta y$ > 1.0'
            
            btext = ''
            if '0b' in cat:
                btext = '0 b-tags'
            elif '1b' in cat:
                btext = '1 b-tag'
            elif '2b' in cat:
                btext = '2 b-tags'
            
#             text = f'data/simulation\n{btext}, {dytext} \n'
            text = f'Preliminary\n{btext}, {dytext} \n'
            
            plotBackgroundEstimate(variable, hdata, hntmj_fixed, httbar, IOV, text, hsignal1000, hsignal2000, hsignal3000, hsignal4000, hsignal5000)
#             if '2016all' in IOV: 
#                 plotBackgroundEstimate(variable, hdata, hntmj_fixed, httbar, IOV, text, hsignal1000, hsignal2000, hsignal3000, hsignal4000)
#             else:
#                 plotBackgroundEstimate(variable, hdata, hntmj_fixed, httbar, IOV, text, hsignal1000, hsignal2000, hsignal3000, hsignal4000)
            

#         plt.savefig(f'images/png/{dirname}/{IOV}/closuretest{signal}_withDM_inclusive.png')
#         plt.savefig(f'images/pdf/{dirname}/{IOV}/closuretest{signal}_withDM_inclusive.pdf')
            print(f'images/png/{dirname}/{IOV}/closuretest_inclusive.png saved\n')

            savefilename = f'images/png/{dirname}/{IOV}/closuretest_withZ10_{cat}.png'
            plt.savefig(savefilename)
            plt.savefig(savefilename.replace('png', 'pdf'))

            plt.show()
            
    for IOV in IOVs:
        for cat in cats:
            print('saving', f'images/png/{dirname}/{IOV}/closuretest_{cat}.png' )       
        print()
        for cat in cats:
            print('saving', f'images/pdf/{dirname}/{IOV}/closuretest_{cat}.pdf')
else:

    for IOV in IOVs:

        for cat in cats:

            signal_cat = label_to_int_dict['2t'+cat]
            pretag_cat = label_to_int_dict['pret'+cat]

            if '2016all' in IOV:

                httbar_apv = getHist(variable, 'TTbar', False, '2016APV', sum_axes=[], integrate_axes={'anacat':signal_cat})
                hcontam_apv = getHist(variable, 'TTbar', True, '2016APV', sum_axes=[], integrate_axes={'anacat':pretag_cat})
                hntmj_apv = getHist(variable, 'JetHT', True, '2016APV', sum_axes=[], integrate_axes={'anacat':pretag_cat})
                hdata_apv = getHist(variable, 'JetHT', False, '2016APV', sum_axes=[], integrate_axes={'anacat':signal_cat})

                httbar_noapv = getHist(variable, 'TTbar', False, '2016', sum_axes=[], integrate_axes={'anacat':signal_cat})
                hcontam_noapv = getHist(variable, 'TTbar', True, '2016', sum_axes=[], integrate_axes={'anacat':pretag_cat})
                hntmj_noapv = getHist(variable, 'JetHT', True, '2016', sum_axes=[], integrate_axes={'anacat':pretag_cat})
                hdata_noapv = getHist(variable, 'JetHT', False, '2016', sum_axes=[], integrate_axes={'anacat':signal_cat})


                httbar = httbar_apv + httbar_noapv
                hcontam = hcontam_apv + hcontam_noapv
                hntmj = hntmj_apv + hntmj_noapv
                hdata = hdata_apv + hdata_noapv
                
                hntmj_fixed = hntmj + -1*hcontam

            else:

                httbar = getHist(variable, 'TTbar', False, IOV, sum_axes=[], integrate_axes={'anacat':signal_cat})
                hcontam = getHist(variable, 'TTbar', True, IOV, sum_axes=[], integrate_axes={'anacat':pretag_cat})
                hntmj = getHist(variable, 'JetHT', True, IOV, sum_axes=[], integrate_axes={'anacat':pretag_cat})
                hdata = getHist(variable, 'JetHT', False, IOV, sum_axes=[], integrate_axes={'anacat':signal_cat})
                
                hntmj_fixed = hntmj + -1*hcontam

            dytext = ''
            if 'cen' in cat:
                dytext = r'$\Delta y$ < 1.0'
            elif 'fwd' in cat:
                dytext = r'$\Delta y$ > 1.0'

            btext = ''
            if '0b' in cat:
                btext = '0 b-tags'
            elif '1b' in cat:
                btext = '1 b-tag'
            elif '2b' in cat:
                btext = '2 b-tags'

#             text = f'data/simulation\n{btext}, {dytext} \n'
            text = f'Preliminary\n{btext}, {dytext} \n'

            hepplot = plotBackgroundEstimate(variable, hdata, hntmj_fixed, httbar, IOV, text)

            savefilename = f'images/png/{dirname}/{IOV}/{variable}_{cat}.png'
            plt.savefig(savefilename)
            plt.savefig(savefilename.replace('png', 'pdf'))

            plt.show()
        
        
        
    # print image file locations
    for IOV in IOVs:
        for cat in cats:
            print('saving', f'images/png/{dirname}/{IOV}/{variable}_{cat}''.png' )       
        print()
        for cat in cats:
            print('saving', f'images/pdf/{dirname}/{IOV}/{variable}_{cat}''.pdf')


In [ ]:
# Signals = {
#     'ZPrime10': ['1000', '2000', '3000', '4000'],
#     'ZPrime30': ['1000', '2000', '3000', '4000'],
#     'ZPrimeDM': ['1000', '1500', '2000', '2500', '3000', '3500', '4000', '4500', '5000'],
#     'RSGluon':  ['1000', '1500', '2000', '2500', '3000', '3500', '4000', '4500', '5000']
# }

# # signal = 'ZPrime30'

# IOV = '2017'

# cats = ['0bcen', '0bfwd', '1bcen', '1bfwd', '2bcen', '2bfwd']
# cat_labels = ['cen0b', 'fwd0b', 'cen1b', 'fwd1b', 'cen2b', 'fwd2b']

# systematics = ['nominal', 'jes', 'jer', 'pileup', 'pdf', 'q2', 'btag', 'prefiring']
# syst_labels = ['nominal']
# for s in systematics:
#     if not 'nominal' in s:
#         syst_labels.append(s+'Down')
#         syst_labels.append(s+'Up')
        
# print(syst_labels)

# savefileheader = '../outputs/combine/categories/TTbarAllHad{}_'.format(IOV.replace('20', '').replace('all',''))
# print(savefileheader)

# froot = uproot.recreate(savefileheader+'CombineRoot_Cat.root')

# variable = 'ttbarmass'



# # for cat, catname in zip(cats, cat_labels):
# #     signal_cat = label_to_int_dict['2t'+cat]
# #     pretag_cat = label_to_int_dict['pret'+cat]
# #     httbar_trial = getHist(variable, 'TTbar', False, '2017', sum_axes=[], integrate_axes={'anacat':signal_cat})
# #     print(httbar_trial)



# for IOV in IOVs:

#     for cat, catname in zip(cats, cat_labels):

#         signal_cat = label_to_int_dict['2t'+cat]
#         pretag_cat = label_to_int_dict['pret'+cat]
        
#         hsignal = {}
#         hsignal['ZPrime10'] = {}
#         hsignal['ZPrime30'] = {}
#         hsignal['ZPrimeDM'] = {}
#         hsignal['RSGluon'] = {}
        
#         for syst in syst_labels:
            
#             if '2016all' in IOV:
                
#                 catsystString = catname+'_'+syst

#                 httbar_apv        = getHist(variable, 'TTbar', False, '2016APV', sum_axes=[], integrate_axes={'anacat':signal_cat, 'systematic':syst})
#                 hcontam_apv       = getHist(variable, 'TTbar', True, '2016APV', sum_axes=[], integrate_axes={'anacat':pretag_cat, 'systematic':syst})
#                 httbar_noapv      = getHist(variable, 'TTbar', False, '2016', sum_axes=[], integrate_axes={'anacat':signal_cat, 'systematic':syst})
#                 hcontam_noapv     = getHist(variable, 'TTbar', True, '2016', sum_axes=[], integrate_axes={'anacat':pretag_cat, 'systematic':syst})
                
#                 print('loading signals...')
#                 for sig in Signals.keys():
#                     for mass in Signals[sig]:
#                         hsignal_apv   = getHist(variable, sig, False, '2016APV', sum_axes=[], integrate_axes={'anacat':signal_cat, 'systematic':syst}, masspoint=mass)
#                         hsignal_noapv = getHist(variable, sig, False, '2016', sum_axes=[], integrate_axes={'anacat':signal_cat, 'systematic':syst}, masspoint=mass)
#                         hsignal[sig][mass] = hsignal_apv + hsignal_noapv
                    
#                 httbar  = httbar_apv + httbar_noapv
#                 hcontam = hcontam_apv + hcontam_noapv
                
#                 print('filling root files...')
#                 if 'nominal' in syst:
#                     syst = ''
#                     catsystString = catname+syst
#                     hntmj_apv   = getHist(variable, 'JetHT', True, '2016APV',   sum_axes=[], integrate_axes={'anacat':pretag_cat, 'systematic':'nominal'})
#                     hdata_apv   = getHist(variable, 'JetHT', False, '2016APV',  sum_axes=[], integrate_axes={'anacat':signal_cat, 'systematic':'nominal'})
#                     hntmj_noapv = getHist(variable, 'JetHT', True, '2016',   sum_axes=[], integrate_axes={'anacat':pretag_cat, 'systematic':'nominal'})
#                     hdata_noapv = getHist(variable, 'JetHT', False, '2016',  sum_axes=[], integrate_axes={'anacat':signal_cat, 'systematic':'nominal'})
                    
#                     hntmj = hntmj_apv + hntmj_noapv
#                     hdata = hdata_apv + hdata_noapv
#                     hntmj_fixed = hntmj + -1*hcontam
#                     print('data_obs_'+catsystString+'\nbkgest_'+catsystString)
#                     froot["data_obs_"+catsystString] = hdata
#                     froot["bkgest_"+catsystString] = hntmj_fixed
                    
#                 print('TTbar_'+catsystString)
#                 froot["TTbar_"+catsystString] = httbar
                
                
#                 for sig in Signals.keys():

#                     if 'RSGluon' not in sig:
#                         [print(sig[:-2]+mass+'_'+sig[-2:]+'_'+catsystString) for mass in Signals[sig]]
#                         for mass in Signals[sig]:
#                             froot[sig[:-2]+mass+'_'+sig[-2:]+'_'+catsystString] = hsignal[sig][mass]
#                     else:
#                         [print(sig+mass+'_'+catsystString) for mass in Signals[sig]]
#                         for mass in Signals[sig]:
#                             froot[sig+mass+'_'+catsystString] = hsignal[sig][mass]

#                 # -- quick nominal plot to see how hists look -- #
# #                 if syst == '':
#                 dytext = ''
#                 if 'cen' in cat:
#                     dytext = r'$\Delta y$ < 1.0'
#                 elif 'fwd' in cat:
#                     dytext = r'$\Delta y$ > 1.0'

#                 btext = ''
#                 if '0b' in cat:
#                     btext = '0 b-tags'
#                 elif '1b' in cat:
#                     btext = '1 b-tag'
#                 elif '2b' in cat:
#                     btext = '2 b-tags'

#                 text = f'data/simulation\n{btext}, {dytext} \n'

#                 plotBackgroundEstimate(variable, hdata, hntmj_fixed, httbar, IOV, text, 
#                                        hsignal['ZPrime10']['1000'], hsignal['ZPrime30']['2000'], hsignal['ZPrimeDM']['3000'], hsignal['ZPrimeDM']['4000'])
#                 plt.show()

#             else:

#                 catsystString = catname+'_'+syst

#                 httbar   = getHist(variable, 'TTbar', False, IOV, sum_axes=[], integrate_axes={'anacat':signal_cat, 'systematic':syst})
#                 hcontam  = getHist(variable, 'TTbar', True, IOV, sum_axes=[], integrate_axes={'anacat':pretag_cat, 'systematic':syst})
                
#                 print('loading signals...')
#                 for sig in Signals.keys():
#                     for mass in Signals[sig]:
#                         hsignal_sigmass   = getHist(variable, sig, False, IOV, sum_axes=[], integrate_axes={'anacat':signal_cat, 'systematic':syst}, masspoint=mass)
#                         hsignal[sig][mass] = hsignal_sigmass
                    
                
#                 print('filling root files...')
#                 if 'nominal' in syst:
#                     syst = ''
#                     catsystString = catname+syst
                    
#                     hntmj = getHist(variable, 'JetHT', True, IOV,   sum_axes=[], integrate_axes={'anacat':pretag_cat, 'systematic':'nominal'})
#                     hdata = getHist(variable, 'JetHT', False, IOV,  sum_axes=[], integrate_axes={'anacat':signal_cat, 'systematic':'nominal'})
#                     hntmj_fixed = hntmj + -1*hcontam
#                     print('data_obs_'+catsystString+'\nbkgest_'+catsystString)
#                     froot["data_obs_"+catsystString] = hdata
#                     froot["bkgest_"+catsystString] = hntmj_fixed
                    
#                 print('TTbar_'+catsystString)
#                 froot["TTbar_"+catsystString] = httbar
                
                
#                 for sig in Signals.keys():

#                     if 'RSGluon' not in sig:
#                         [print(sig[:-2]+mass+'_'+sig[-2:]+'_'+catsystString) for mass in Signals[sig]]
#                         for mass in Signals[sig]:
#                             froot[sig[:-2]+mass+'_'+sig[-2:]+'_'+catsystString] = hsignal[sig][mass]
#                     else:
#                         [print(sig+mass+'_'+catsystString) for mass in Signals[sig]]
#                         for mass in Signals[sig]:
#                             froot[sig+mass+'_'+catsystString] = hsignal[sig][mass]

#                 # -- quick nominal plot to see how hists look -- #
# #                 if syst == '':
#                 dytext = ''
#                 if 'cen' in cat:
#                     dytext = r'$\Delta y$ < 1.0'
#                 elif 'fwd' in cat:
#                     dytext = r'$\Delta y$ > 1.0'

#                 btext = ''
#                 if '0b' in cat:
#                     btext = '0 b-tags'
#                 elif '1b' in cat:
#                     btext = '1 b-tag'
#                 elif '2b' in cat:
#                     btext = '2 b-tags'

#                 text = f'data/simulation\n{btext}, {dytext} \n'

#                 plotBackgroundEstimateNoData(variable, hntmj_fixed, httbar, IOV, text, 
#                                        hsignal['RSGluon']['2000'], hsignal['ZPrime10']['2000'], hsignal['ZPrime30']['2000'], hsignal['ZPrimeDM']['2000'])
#                 plt.show()
                    
# froot.close()            
            
            
            